# Phase 1 (Suite) : Modèle Avancé (XGBoost) 🚀

Objectif : 
1. Entraîner le vrai modèle Champion : **XGBoost**.
2. Sauvegarder ce modèle et le Preprocessor pour la production (API).

*Pourquoi XGBoost ? C'est le State-of-the-Art pour les données tabulaires (Excel/CSV), extrêmement rapide, et 100% compatible avec l'explicabilité SHAP.*

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from loguru import logger
from src.data.loader import DataLoader
from src.data.preprocessor import FraudPreprocessor
from src.models.trainer import ModelTrainer
from src.models.evaluator import ModelEvaluator

## 1. Préparation des données (Identique aux baselines)

In [ ]:
loader = DataLoader()
df = loader.load()

cols_to_drop = ['organization', 'transaction_id', 'user_id', 'transaction_timestamp']
df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

X_train, X_val, X_test, y_train, y_val, y_test = loader.get_splits(df_clean)

preprocessor = FraudPreprocessor(smote_sampling_strategy=0.2)
X_train_proc, y_train_proc = preprocessor.fit_transform_train(X_train, y_train)
X_val_proc = preprocessor.transform(X_val)

# 🔥 SAUVEGARDE DU PREPROCESSOR POUR LA PRODUCTION 🔥
preprocessor.save()

## 2. Entraînement de XGBoost
Notre implémentation dans `trainer.py` calcule automatiquement le paramètre `scale_pos_weight` pour compenser le déséquilibre restant !

In [ ]:
evaluator = ModelEvaluator()

logger.info("Entraînement de XGBoost...")
# On passe X_val pour faire de l'Early Stopping (éviter le sur-apprentissage)
xgb_trainer = ModelTrainer(model_name="xgboost")
xgb_trainer.fit(X_train_proc, y_train_proc, X_val=X_val_proc, y_val=y_val)

y_val_prob_xgb = xgb_trainer.predict_proba(X_val_proc)[:, 1]

## 3. Évaluation du Champion

In [ ]:
metrics_xgb = evaluator.evaluate(y_val, y_val_prob_xgb, model_name="XGBoost")
fig = evaluator.plot_precision_recall_curve(y_val, y_val_prob_xgb, model_name="XGBoost")

# Cherchons le seuil de décision optimal (qui maximise le compromis Précision/Recall)
best_threshold = evaluator.find_best_threshold(y_val, y_val_prob_xgb, metric="f1")

## 4. Matrice de Confusion avec le meilleur seuil

In [ ]:
evaluator_optimal = ModelEvaluator(threshold=best_threshold)
cm = evaluator_optimal.confusion_matrix_df(y_val, y_val_prob_xgb)
display(cm)

## 5. Sauvegarde du Modèle Champion 🏆
C'est ce fichier qui sera chargé par l'API FastAPI !

In [ ]:
xgb_trainer.save()